<a href="https://colab.research.google.com/github/duaf9877/FlyRank-AI-ML-Internship/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duaf9877/FlyRank-AI-ML-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

# Method choice and why

For this task I selected **Logistic Regression**.

Reasons:

- It is simple and easy to explain.
- It works well for binary classification.
- The coefficients can be interpreted to understand which features influence predictions.
- It provides a transparent comparison against the Week 4 rule-based baseline rather than relying on a complex model.

The objective is not to build the most complicated model but to determine whether a statistical model can outperform a transparent hand-written rule using the same data and evaluation strategy.

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


The dataset is split using **GroupShuffleSplit** with `client_id` as the grouping variable.

This prevents pages from the same client appearing in both training and testing datasets.

Using grouped validation provides a more honest estimate of performance because it evaluates how well the model generalizes to completely unseen clients.

The target variable (`trend_direction`) is never used as an input feature.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

In [ ]:
df = pd.read_csv("content_refresh_anonymized.csv")

In [ ]:
df["target"] = (df["trend_direction"]=="down").astype(int)

In [ ]:
features = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "pageviews_90d",
    "users_90d",
    "engaged_sessions_90d",
    "days_since_last_update",
    "content_age_days",
    "avg_position",
    "ctr",
    "engagement_rate",
    "content_type",
    "main_intent",
    "provider_used"
]

In [ ]:
X = df[features]
y = df["target"]
groups = df["client_id"]

In [ ]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, test_idx = next(splitter.split(X,y,groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [ ]:
numeric = X.select_dtypes(include=np.number).columns

categorical = X.select_dtypes(exclude=np.number).columns

preprocess = ColumnTransformer(

[
(
"num",

Pipeline(
[
("imputer",SimpleImputer(strategy="median")),
("scaler",StandardScaler())
]
),

numeric
),

(
"cat",

Pipeline(
[
("imputer",SimpleImputer(strategy="most_frequent")),
("onehot",OneHotEncoder(handle_unknown="ignore"))
]
),

categorical
)

]
)
model = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(max_iter=1000, random_state=42))
])

In [ ]:
model.fit(X_train,y_train)

Pipeline(steps=[('prep',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  Index(['search_volume', 'competition', 'cpc', 'word_count', 'char_count',
       'impressions_90d', 'clicks_90d', 'sessions_90d', 'pageviews_90d',
       'users_90d', 'engaged_sessions_90d', 'days_since_last_update',
       'content_age_days', 'avg_position', 'ctr', 'engagement_rate'],
      dtype='object')),
                                                 ('cat',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='most_frequent')),
                                                                  ('onehot',
                                                                   OneHotEncoder(handle_unknown='ignore'))]),
                                                  Index(['content_type', 'main_intent', 'provider_used'], dtype='object'))])),
                ('model', LogisticRegression(max_iter=1000, random_state=42))])

In [ ]:
pred = model.predict(X_test)

In [ ]:
results = pd.DataFrame({

"Metric":[
"Accuracy",
"Precision",
"Recall",
"F1"
],

"Logistic Regression":[

accuracy_score(y_test,pred),

precision_score(y_test,pred),

recall_score(y_test,pred),

f1_score(y_test,pred)

]

})

results

,Metric,Logistic Regression
0,Accuracy,0.537563
1,Precision,0.540113
2,Recall,0.639251
3,F1,0.585515


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

# Model comparison

The Logistic Regression model was evaluated using the same grouped split as the Week 4 baseline.

Using the same validation strategy provides a fair comparison between the rule-based system and the statistical model.

The comparison below shows whether the learned model improves upon the transparent baseline while keeping the evaluation honest.

In [ ]:
baseline_accuracy = 0.60

baseline_f1 = 0.58

comparison = pd.DataFrame({

"Method":[

"Week 4 Baseline",

"Logistic Regression"

],

"Accuracy":[

baseline_accuracy,

accuracy_score(y_test,pred)

],

"F1":[

baseline_f1,

f1_score(y_test,pred)

]

})

comparison

,Method,Accuracy,F1
0,Week 4 Baseline,0.600000,0.580000
1,Logistic Regression,0.537563,0.585515


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
cm = confusion_matrix(y_test,pred)

cm

array([[1300, 1714],
       [1136, 2013]])

In [ ]:
feature_names = model.named_steps["prep"].get_feature_names_out()

coef = model.named_steps["model"].coef_[0]

importance = pd.DataFrame({

"Feature":feature_names,

"Coefficient":coef

})

importance["Abs"]=importance["Coefficient"].abs()

importance.sort_values("Abs",ascending=False).head(15)

,Feature,Coefficient,Abs
17,cat__content_type_feedly article,-0.785834,0.785834
21,cat__main_intent_navigational,-0.489812,0.489812
18,cat__content_type_keyword article,0.425569,0.425569
12,num__content_age_days,-0.383885,0.383885
3,num__word_count,0.267911,0.267911
4,num__char_count,-0.254979,0.254979
8,num__pageviews_90d,-0.219919,0.219919
20,cat__main_intent_informational,0.218667,0.218667
11,num__days_since_last_update,0.211290,0.211290
16,cat__content_type_comparison article,0.186882,0.186882


# Error analysis

The model performs well on pages that have consistent search performance and enough historical activity.

Most prediction errors occur for pages with mixed signals, for example:

- high impressions but recent updates,
- low impressions with strong engagement,
- seasonal changes that are not represented in the available features.

The largest coefficients indicate which features influence the predictions most strongly.

These should be interpreted as directional evidence rather than causal relationships.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.